In [6]:
%reload_ext autoreload
%autoreload 2
# 上面这俩用于自动重载

import sqlite3 # 先导入内置的模块

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, quality, viz
from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量

viz.apply_theme(theme="default")

con = sqlite3.connect(config.DB_PATH) # 连接数据库
DAY = config.ANALYSIS_DAY
NB = "D03_信令数据洞察"
SOURCE = f"{config.SOURCE_NOTE}:脚本:{NB} 制图:{config.AUTHOR}"

In [7]:
raw = pd.read_sql("SELECT * FROM ods_signal", con)

raw['ts'] = pd.to_datetime(raw['timestamp'], unit="ms") + \
    pd.Timedelta(hours = config.UTC_OFFSET_HOURS)
raw.shape

(532828, 11)

In [8]:
valid = raw.dropna(subset=['imsi'])
valid = valid[valid['imsi'].str.fullmatch(r"\d{18}")
              & (valid['phone'] != "6666")]
# 只看那个imsi 配过那个手机号，这组对应关系，同一对出现多少次不重要，所以要先去重
pairs = valid[['imsi','phone']].drop_duplicates()
# 裁剪出前15位，作为另外一种认证方式
pairs['imsi15']  = pairs['imsi'].str[:15]

per18 = pairs.groupby("imsi")['phone'].nunique()
per15 = pairs.groupby('imsi15')['phone'].nunique()

key_check = pd.DataFrame({
    "分组方式": ['按前15位',"按完整18位"],
    "ID个数": [per15.shape[0], per18.shape[0]],
    "对应多个手机号的ID": [(per15 > 1).sum(), (per18 > 1).sum()]
})
key_check['占比'] = (key_check['对应多个手机号的ID'] / key_check['ID个数']).round(3)
key_check.to_sql("ads_key_check", con, if_exists="replace",index=False)

2

In [9]:
multi = per15[per15 > 1].index
affected = pairs.loc[pairs['imsi15'].isin(multi),'imsi'].nunique()
suffuix_kinds = pairs['imsi'].str[-3:].nunique()
print(f"对应多个手机号的前15位分组："
    f"{len(multi)}组，里面装着{affected:,}个不同的18位IMSI,"
    f"占全部用户的{affected / len(per18):.1%}")
print(f"后三位一共有{suffuix_kinds}种取值")

对应多个手机号的前15位分组：729组，里面装着1,707个不同的18位IMSI,占全部用户的30.2%
后三位一共有995种取值


In [10]:
k = pd.read_sql("SELECT * FROM ads_key_check", con)
raws = [(f"按前 15 位分组\n（{k.loc[0, 'ID个数']:,} 组）",
         k.loc[0, "对应多个手机号的ID"],
         k.loc[0, "ID个数"], "组对应多个手机号"),
        (f"被牵连的用户\n（{len(per18):,} 个 18 位 IMSI）", affected, len(per18),
         "个用户会被错误合并"),
        (f"按完整 18 位\n（{k.loc[1, 'ID个数']:,} 个）", k.loc[1, "对应多个手机号的ID"],
         k.loc[1, "ID个数"], "个对应多个手机号")]
raws

[('按前 15 位分组\n（4,667 组）', np.int64(729), np.int64(4667), '组对应多个手机号'),
 ('被牵连的用户\n（5,645 个 18 位 IMSI）', 1707, 5645, '个用户会被错误合并'),
 ('按完整 18 位\n（5,645 个）', np.int64(0), np.int64(5645), '个对应多个手机号')]